# RG3 정상 관계 잔차의 교차적합

82번 확장: 관계 모델이 본 정상 샷의 잔차는 작고, 보지 않은 가스 잔차는 커지는 자기학습 편향을 완화한다. 분류기 학습용 잔차는 학습 자료 내부에서 해당 관측구간 정상도 제외한 관계 모델로 계산. 외부 평가 잔차는 외부 평가를 제외한 모든 학습 정상 관계로 계산. 목표 센서 단위 차이를 IQR로 먼저 보정해 압력 하나가 다출력 회귀를 지배하는 문제를 완화.

RG3 RH만 정상601/가스19. CN7 미사용. 동일 7개 관측구간 중첩검증. 완전 새 독립 검증 아님. 외부 평가 라벨/센서는 관계 모델·전처리·분류·임계값 학습에서 제외. 모델 개발 자료에서 개선해도 배포 성능을 보장하지 않음.


In [1]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import ExtraTreesRegressor,RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s=''.join(json.loads((root/'notebooks/67_rg3_recall_first_nested_mode_ensemble.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s.split('F=pd.DataFrame')[0])
assert len(d)==620 and y.sum()==19
inputs=['Max_Injection_Speed','Average_Screw_RPM','Barrel_Temperature_2','Barrel_Temperature_5','Plasticizing_Position','Filling_Time']
targets=['Max_Injection_Pressure','Max_Switch_Over_Pressure','Cushion_Position','Plasticizing_Time']
stdcols=[x+'__past10_std' for x in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']]
memo={};audit=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 normal=ref[y[ref]==0]
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(d.iloc[normal][inputs+targets+stdcols])
 tr=im.transform(d.iloc[ref][inputs+targets+stdcols]);te=im.transform(d.iloc[q][inputs+targets+stdcols])
 nt=tr[y[ref]==0]
 scale=np.maximum(np.quantile(nt[:,6:10],.75,axis=0)-np.quantile(nt[:,6:10],.25,axis=0),1e-6)
 def fit_relation(mask):
  z=tr[mask]
  return ExtraTreesRegressor(n_estimators=60,min_samples_leaf=12,max_depth=5,random_state=42,n_jobs=2).fit(z[:,:6],z[:,6:10]/scale)
 relation=fit_relation(y[ref]==0)
 rt=np.empty((len(ref),4))
 for ib in groups.iloc[ref].unique():
  held=groups.iloc[ref].eq(ib).to_numpy()
  local=fit_relation((~held)&(y[ref]==0))
  rt[held]=tr[held,6:10]/scale-local.predict(tr[held,:6])
 rq=te[:,6:10]/scale-relation.predict(te[:,:6])
 features=[(tr[:,10:],te[:,10:]),(np.c_[rt,np.abs(rt)],np.c_[rq,np.abs(rq)])]
 scores=[]
 for ft,fq in features:
  vals=[]
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   for seed in [42,7]:
    if len(np.unique(y[ref]))<2:vals.append(np.zeros(len(q)));continue
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
    vals.append(m.fit(ft,y[ref]).predict_proba(fq)[:,1])
  scores.append(np.mean(vals,axis=0))
 memo[key]=np.column_stack([scores[0],scores[1],np.maximum(*scores)])
 audit.append(dict(train=len(ref),query=len(q),normal_relation_training=len(normal),gas_training=int(y[ref].sum()),train_blocks=','.join(map(str,groups.iloc[ref].unique())),query_blocks=','.join(map(str,groups.iloc[q].unique()))))
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),3),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['std4_reference','crossfit_residual8','std4_crossfit_max_ensemble']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   pp=inner[:,k]>cut;tn,fp,fn,tp=confusion_matrix(y[ref],pp,labels=[0,1]).ravel()
   rec=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((rec,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut),score=float(out[j,k]),threshold=cut))
 chosen=max(candidates,key=lambda x:x[:3])
 choices.append(dict(block=int(b),selected=chosen[3],budget=chosen[4],inner_recall=chosen[0],inner_F1=chosen[1]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(chosen[5][j]),score=np.nan,threshold=np.nan))
 print('Finished RG3 conditional relationship block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget),g in p.groupby(['model','budget']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn),AP=average_precision_score(g.actual,g.score) if g.score.notna().all() else np.nan))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(tables/f'rg3_crossfit_residual_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))



Finished RG3 conditional relationship block 1


Finished RG3 conditional relationship block 2


Finished RG3 conditional relationship block 3


Finished RG3 conditional relationship block 4


Finished RG3 conditional relationship block 5


Finished RG3 conditional relationship block 6


Finished RG3 conditional relationship block 7


                      model  budget  TP  FP  FN  TN   recall       F1       AP
         crossfit_residual8    0.02   0  14  19 587 0.000000 0.000000 0.030321
         crossfit_residual8    0.05   1  57  18 544 0.052632 0.025974 0.030321
         crossfit_residual8    0.10   4 106  15 495 0.210526 0.062016 0.030321
inner_recall_first_selected    0.00   3  63  16 538 0.157895 0.070588      NaN
 std4_crossfit_max_ensemble    0.02   3  26  16 575 0.157895 0.125000 0.056279
 std4_crossfit_max_ensemble    0.05   3  33  16 568 0.157895 0.109091 0.056279
 std4_crossfit_max_ensemble    0.10   3  59  16 542 0.157895 0.074074 0.056279
             std4_reference    0.02   3  26  16 575 0.157895 0.125000 0.061578
             std4_reference    0.05   3  33  16 568 0.157895 0.109091 0.061578
             std4_reference    0.10   3  67  16 534 0.157895 0.067416 0.061578
